# 04.P · Solution: a leak-free churn pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/project/solution.ipynb)

## 0. Load and clean

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])
print(f"{len(sales):,} clean transactions from {sales['InvoiceDate'].min():%Y-%m-%d} to {sales['InvoiceDate'].max():%Y-%m-%d}")

392,693 clean transactions from 2010-12-01 to 2011-12-09


## 1. Customer table for a cut-off date

In [2]:
def build_table(cutoff):
    """One row per customer active before `cutoff`.
    Features use only orders BEFORE the cut-off; churned = no order in the 3 months AFTER it."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=3))]
    t = past.groupby("Customer ID").agg(
        orders=("Invoice", "nunique"), spend=("Revenue", "sum"), products=("StockCode", "nunique"),
        first=("InvoiceDate", "min"), last=("InvoiceDate", "max"), country=("Country", lambda s: s.mode()[0]))
    t["tenure_days"] = (cutoff - t["first"]).dt.days
    t["days_since_last"] = (cutoff - t["last"]).dt.days
    t["avg_basket"] = t["spend"] / t["orders"]
    recent = past[past["InvoiceDate"] >= cutoff - pd.Timedelta(days=90)].groupby("Customer ID")["Invoice"].nunique()
    t["orders_last_90d"] = recent.reindex(t.index).fillna(0)
    t["churned"] = (~t.index.isin(future["Customer ID"].unique())).astype(int)
    return t.drop(columns=["first", "last"])

## 2. Time-based split

In Module 03 we split *customers* at random. That is fine for learning, but it is not how the model will be used. On 1 September ShopWave will score its customers with a model trained on the past. So we do exactly that: learn from what was known on **1 June** and what happened over the summer, and judge the model on **1 September** and what happened in the autumn.

In [3]:
train = build_table("2011-06-01")      # learn from: features to 1 Jun, outcome Jun-Aug
test = build_table("2011-09-01")       # judge on:   features to 1 Sep, outcome Sep-Nov
num_cols = ["orders", "spend", "products", "tenure_days", "days_since_last", "avg_basket", "orders_last_90d"]
cat_cols = ["country"]
X_train, y_train = train[num_cols + cat_cols], train["churned"]
X_test, y_test = test[num_cols + cat_cols], test["churned"]
for name, t in [("train (cut-off 1 Jun)", train), ("test  (cut-off 1 Sep)", test)]:
    print(f"{name}: {len(t):,} customers, churn rate {t['churned'].mean():.1%}")

train (cut-off 1 Jun): 2,718 customers, churn rate 50.4%
test  (cut-off 1 Sep): 3,317 customers, churn rate 43.6%


The churn rate is different: 50.4% over the summer, 43.6% in the autumn run-up to Christmas. A model trained in June meets a slightly different world in September. Random splits hide this; time-based splits show it.

## 3. A quick look (training table only)

In [4]:
band = pd.cut(train["days_since_last"], [-1, 30, 90, 180, 400], labels=["0-30", "31-90", "91-180", "181+"])
train.groupby(band, observed=True)["churned"].agg(["count", "mean"]).round(3)

,count,mean
days_since_last,,
0-30,1056,0.341
31-90,922,0.538
91-180,712,0.685
181+,28,0.893


Recency is strongly related to churn in the summer table too, so the main signal should carry over to September.

## 4. Preprocessing

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

def make_prep():
    numeric = make_pipeline(SimpleImputer(strategy="median"),
                            FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler())
    categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                                OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                                              sparse_output=False))
    return ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)])

## 5. Compare three models with cross-validation (training table only)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
skf = StratifiedKFold(5, shuffle=True, random_state=42)
candidates = {"logistic regression": LogisticRegression(max_iter=2000),
              "KNN (k=101)": KNeighborsClassifier(n_neighbors=101),
              "tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=42)}
rows = []
for name, model in candidates.items():
    s = cross_val_score(Pipeline([("prep", make_prep()), ("model", model)]), X_train, y_train, cv=skf, scoring="roc_auc")
    rows.append({"model": name, "CV AUC mean": s.mean(), "CV AUC std": s.std()})
pd.DataFrame(rows).round(3)

,model,CV AUC mean,CV AUC std
0,logistic regression,0.752,0.020
1,KNN (k=101),0.755,0.021
2,tree (depth 5),0.718,0.022


## 6. Choose and tune

KNN and logistic regression are within each other's error bars; the tree is clearly behind. We choose **logistic regression**: same accuracy, and its coefficients can be explained to the retention team, it is fast, and it is easy to deploy. Now tune its regularisation strength `C` by CV (still training data only).

In [7]:
for C in [0.01, 0.03, 0.1, 0.3, 1, 3]:
    p = Pipeline([("prep", make_prep()), ("model", LogisticRegression(C=C, max_iter=2000))])
    s = cross_val_score(p, X_train, y_train, cv=skf, scoring="roc_auc")
    print(f"C={C:<5} CV AUC {s.mean():.3f} ± {s.std():.3f}")

C=0.01  CV AUC 0.754 ± 0.022
C=0.03  CV AUC 0.753 ± 0.021
C=0.1   CV AUC 0.753 ± 0.020
C=0.3   CV AUC 0.752 ± 0.020
C=1     CV AUC 0.752 ± 0.020


C=3     CV AUC 0.752 ± 0.019


The scores are flat across C: the model is not overfitting, so regularisation strength barely matters (04.9: this is a bias-limited model). We keep the default C = 1.

## 7. The one final test

In [8]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
final = Pipeline([("prep", make_prep()), ("model", LogisticRegression(C=1, max_iter=2000))])
final.fit(X_train, y_train)
p = final.predict_proba(X_test)[:, 1]
print(f"test AUC      {roc_auc_score(y_test, p):.3f}")
print(f"test accuracy {accuracy_score(y_test, p > 0.5):.3f}  (baseline 'nobody churns': {1 - y_test.mean():.3f})")
print("confusion matrix [rows: true stay / churn; cols: predicted stay / churn]")
print(confusion_matrix(y_test, p > 0.5))
top500 = y_test.to_numpy()[np.argsort(-p)[:500]].mean()
print(f"churn rate among the 500 highest-risk customers: {top500:.1%}  (overall {y_test.mean():.1%})")

test AUC      0.741
test accuracy 0.674  (baseline 'nobody churns': 0.564)
confusion matrix [rows: true stay / churn; cols: predicted stay / churn]
[[1189  682]
 [ 401 1045]]
churn rate among the 500 highest-risk customers: 66.8%  (overall 43.6%)


## 8. Leakage audit

| Risk | How this notebook handles it |
|---|---|
| Features using data after the cut-off | `build_table` computes every feature from `past`, which stops strictly before the cut-off. |
| Test outcomes used for training | The training outcome window is June–August; it ends exactly where the test features begin. September–November outcomes are never seen before the final cell. Some customers appear in both tables, which is fine: different dates, different labels, as in real use. |
| Preprocessing fitted on test or held-out rows | Imputer, log, scaler and encoder live inside the `Pipeline`, refitted in every CV fold and fitted only on the training table at the end. |
| Model chosen by looking at the test score | Model and C were chosen with 5-fold CV on the training table. The test table was used once. |
| A feature that is really the target in disguise | No feature uses the outcome window. `orders_last_90d` looks *back* 90 days from the cut-off, never forward. |

## 9. Conclusion

A logistic regression trained only on what ShopWave knew on 1 June ranks September customers by churn risk with a test AUC of **0.741**, close to its cross-validated estimate of 0.752 on the training data, so the evaluation is honest rather than lucky. Accuracy at a 0.5 threshold is 0.674 against 0.564 for "nobody churns". The model over-predicts churn (682 loyal customers flagged) because it learnt in a summer when half of customers churned, while in the autumn only 44% did; its probabilities are better used for **ranking** than as yes/no answers. Among the 500 customers it rates most at risk, **66.8% really churned**, against 43.6% overall, so that is where the retention budget should go. On 1 September we would score every active customer and send win-back offers from the top of the list down, at the cost-based threshold from lesson 03.7, ideally retraining on the most recent quarter first. The next improvement is not more tuning (CV scores are flat across C) but more information or more flexible models: Module 05's ensembles, and features such as product categories or return history.